# 不适用@tool的方式定义工具

## 1、举例

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

MODEL_NAME = "deepseek-flash"
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

# 2、声明一个函数（工具）
def get_weather(city: str):
    return f"{city}天气晴朗~~"

# 3、将函数绑定在模型上
model_with_tools = model.bind_tools([get_weather])

# 4、调用模型
response = model_with_tools.invoke("北京天气怎么样")
rprint(response)


AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': "The user is asking about Beijing's weather. Let me call the weather tool."
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 55,
            'prompt_tokens': 285,
            'total_tokens': 340,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 16,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 157
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '75b24f1a-3875-488f-9b40-101318f3623d',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a13ebd-722a-77e0-95bd-7985dc9966aa-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_OdCtv2ZKtfJwJug2KIZv9059',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 285,
        'output_tokens': 55,
        'total_tokens': 340,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 16}
    }
)

## 2、工具描述的各部分详解

## 2.1 了解convert_to_openai_tool
执行 model.bind_tools([get_weather]) ，底层最终会调用 convert_to_openai_tool 生成工具描述。所
以我们可以直接调用后者查看解析后的工具描述。


In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str):
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2 description说明

In [1]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str):
    """
    查询城市天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.3 参数说明（Google风格）

In [2]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str):
    """
    查询城市天气

    Args:
        city:具体的城市

    Returns:
        返回城市的天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市天气',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 参数类型的说明

举例1：

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str):
    """
    查询城市天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

举例2：如下的代码运行会报错

要求：如果在docstring中声明了参数的描述，则必须要再函数声明参数的类型

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city):
    """
    查询城市天气

    Args:
        city:具体的城市
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

## 2.5 参数默认值说明

一旦参数设置的默认值，则打印的结果中的required字段中就不再包含此参数

In [4]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city : str = "北京"):
    """
    查询城市天气

    Args:
        city:具体的城市
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市天气',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例2：

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(dt : str ,city : str = "北京"):
    """
    查询城市天气

    Args:
        city:具体的城市
        dt : 时间
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市天气',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}